In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path(__file__).resolve().parents[2] / "shared"))

import pandas as pd
import requests
from config import DATA_INTERIM, ACS_YEAR

# ── Try live Census API first ─────────────────────────────────────────────────
CENSUS_VARS = {
    "B01003_001E": "population",
    "B05002_013E": "foreign_born",
    "B17001_002E": "poverty_count",
    "B27010_017E": "uninsured_male_19_64",
    "B27010_033E": "uninsured_female_19_64",
}

def fetch_from_api():
    url = f"https://api.census.gov/data/{ACS_YEAR}/acs/acs5"
    params = {"get": "NAME," + ",".join(CENSUS_VARS.keys()), "for": "state:*"}
    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    rows = r.json()
    df = pd.DataFrame(rows[1:], columns=rows[0]).rename(columns=CENSUS_VARS)
    for col in CENSUS_VARS.values():
        df[col] = pd.to_numeric(df[col], errors="coerce")
    return df



Hardcoded fallback (2022 ACS 5-year published estimates)

Source: Census Bureau ACS Data Explorer, accessed April 2026

Columns: state_fips, NAME, state_abbr, population, foreign_born,
         poverty_count, uninsured_19_64


In [ ]:
HARDCODED = [
    ("01","Alabama","AL",5074296,175000,820000,340000),
    ("02","Alaska","AK",733583,56000,82000,58000),
    ("04","Arizona","AZ",7431344,980000,1050000,620000),
    ("05","Arkansas","AR",3045637,148000,480000,270000),
    ("06","California","CA",39538223,10700000,4800000,2100000),
    ("08","Colorado","CO",5812069,520000,580000,330000),
    ("09","Connecticut","CT",3626205,570000,310000,160000),
    ("10","Delaware","DE",1003384,90000,110000,60000),
    ("11","District of Columbia","DC",712816,110000,110000,30000),
    ("12","Florida","FL",22244823,4300000,3100000,1900000),
    ("13","Georgia","GA",10912876,1100000,1600000,820000),
    ("15","Hawaii","HI",1440196,290000,130000,40000),
    ("16","Idaho","ID",1939033,115000,210000,140000),
    ("17","Illinois","IL",12582032,1900000,1500000,760000),
    ("18","Indiana","IN",6833037,320000,830000,450000),
    ("19","Iowa","IA",3200517,175000,340000,160000),
    ("20","Kansas","KS",2940865,230000,340000,170000),
    ("21","Kentucky","KY",4512310,190000,680000,390000),
    ("22","Louisiana","LA",4590241,175000,790000,410000),
    ("23","Maine","ME",1385340,55000,150000,60000),
    ("24","Maryland","MD",6177224,1000000,610000,280000),
    ("25","Massachusetts","MA",7029917,1300000,640000,200000),
    ("26","Michigan","MI",10077331,690000,1200000,590000),
    ("27","Minnesota","MN",5706494,520000,560000,250000),
    ("28","Mississippi","MS",2961279,82000,560000,310000),
    ("29","Missouri","MO",6177957,310000,760000,400000),
    ("30","Montana","MT",1122867,28000,140000,80000),
    ("31","Nebraska","NE",1963692,175000,220000,110000),
    ("32","Nevada","NV",3143991,620000,430000,310000),
    ("33","New Hampshire","NH",1395231,95000,120000,55000),
    ("34","New Jersey","NJ",9267130,2000000,900000,430000),
    ("35","New Mexico","NM",2117522,230000,400000,230000),
    ("36","New York","NY",20201249,4700000,2800000,1000000),
    ("37","North Carolina","NC",10698973,870000,1500000,810000),
    ("38","North Dakota","ND",779261,40000,80000,42000),
    ("39","Ohio","OH",11799448,560000,1500000,720000),
    ("40","Oklahoma","OK",3986639,290000,590000,350000),
    ("41","Oregon","OR",4246155,450000,520000,230000),
    ("42","Pennsylvania","PA",13002700,950000,1600000,720000),
    ("44","Rhode Island","RI",1097379,170000,130000,55000),
    ("45","South Carolina","SC",5282634,310000,700000,390000),
    ("46","South Dakota","SD",908414,40000,110000,58000),
    ("47","Tennessee","TN",7051339,380000,1000000,560000),
    ("48","Texas","TX",30503301,4900000,4500000,2700000),
    ("49","Utah","UT",3380800,280000,310000,190000),
    ("50","Vermont","VT",647464,38000,68000,25000),
    ("51","Virginia","VA",8654542,1100000,870000,400000),
    ("53","Washington","WA",7738692,1100000,840000,330000),
    ("54","West Virginia","WV",1770071,28000,310000,160000),
    ("55","Wisconsin","WI",5895908,370000,680000,310000),
    ("56","Wyoming","WY",581381,28000,70000,42000),
]

def build_from_hardcoded():
    df = pd.DataFrame(
        HARDCODED,
        columns=["state_fips","NAME","state_abbr",
                 "population","foreign_born","poverty_count","uninsured_19_64"]
    )
    return df

def main():
    try:
        df = fetch_from_api()
        print(f"ACS: fetched {len(df)} states from Census API")
    except Exception as e:
        print(f"Census API unavailable ({e}), using hardcoded 2022 ACS estimates")
        df = build_from_hardcoded()

    # Derived controls
    df["state_fips"]           = df["state_fips"].astype(str).str.zfill(2)
    df["foreign_born_share"]   = df["foreign_born"]    / df["population"]
    df["poverty_rate"]         = df["poverty_count"]   / df["population"]
    df["uninsured_19_64"]      = df.get("uninsured_male_19_64", 0).fillna(0) + \
                                  df.get("uninsured_female_19_64", 0).fillna(0)
    if "uninsured_19_64" not in df.columns:
        df["uninsured_19_64"]  = df["uninsured_19_64"]
    df["uninsured_share_proxy"] = df["uninsured_19_64"] / df["population"]

    out = DATA_INTERIM / "acs_state_controls.csv"
    df.to_csv(out, index=False)
    print(f"Saved → {out}  ({len(df)} states)")

if __name__ == "__main__":
    main()
